In [6]:
!pip install scikit-learn


  Using cached scikit_learn-1.7.1-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.16.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.7.1-cp313-cp313-win_amd64.whl (8.7 MB)
Using cached joblib-1.5.1-py3-none-any.whl (307 kB)
Using cached scipy-1.16.1-cp313-cp313-win_amd64.whl (38.5 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ------------------

In [13]:
!pip install tensorflow
!pip install keras

In [7]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras import backend as K
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib
import os

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.20.0


In [8]:
# Load the dataset
df = pd.read_csv('data/valid_gear_designs.csv')

# Display the first few rows and basic info
print("Dataset Head:")
print(df.head())
print("\nDataset Info:")
df.info()

Dataset Head:
   power_kw  pinion_speed_rpm  ratio  module  pinion_teeth  face_width_mm  \
0        10              1000    1.5     2.0            32            100   
1        10              1000    1.5     2.0            33            100   
2        10              1000    1.5     2.0            34            100   
3        10              1000    1.5     2.0            35             90   
4        10              1000    1.5     2.0            35            100   

   pressure_angle_deg  min_safety_factor  material_name  \
0                20.0                1.5  HardenedSteel   
1                20.0                1.5  HardenedSteel   
2                20.0                1.5  HardenedSteel   
3                20.0                1.5  HardenedSteel   
4                20.0                1.5  HardenedSteel   

   material_allowable_bending_stress_mpa  \
0                                    250   
1                                    250   
2                                   

In [9]:
# --- 1. Separate conditions (inputs) from design data (outputs) ---
# The model will learn to generate 'design_data' given the 'conditions'.

# Define the columns for each part
condition_cols = [
    'power_kw', 'pinion_speed_rpm', 'ratio', 'material_name',
    'material_allowable_bending_stress_mpa', 'material_allowable_contact_stress_mpa'
]

design_data_cols = [
    'module', 'pinion_teeth', 'face_width_mm'
]

conditions_df = df[condition_cols]
design_df = df[design_data_cols]


# --- 2. Create Preprocessing Pipelines ---
# We use scikit-learn's ColumnTransformer to handle different data types.
# - OneHotEncoder for the 'material_name' (categorical).
# - MinMaxScaler for all other numerical columns to scale them between 0 and 1.

# Identify categorical and numerical columns in the conditions
categorical_features_cond = ['material_name']
numerical_features_cond = [col for col in condition_cols if col not in categorical_features_cond]

# Create the transformer for conditions
# 'passthrough' means we keep the other columns as they are (after scaling)
preprocessor_conditions = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), numerical_features_cond),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_cond)
    ])

# Create a simple scaler for the design data (it's all numerical)
preprocessor_design = MinMaxScaler()


# --- 3. Fit and Transform the Data ---
X_conditions = preprocessor_conditions.fit_transform(conditions_df)
X_design = preprocessor_design.fit_transform(design_df)

print("Shape of processed conditions:", X_conditions.shape)
print("Shape of processed design data:", X_design.shape)
print("\nPreprocessing complete.")


# --- 4. Save the Preprocessors (VERY IMPORTANT) ---
# We need to save these fitted objects to use them later in our application.
output_dir = 'models'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

joblib.dump(preprocessor_conditions, os.path.join(output_dir, 'preprocessor_conditions.joblib'))
joblib.dump(preprocessor_design, os.path.join(output_dir, 'preprocessor_design.joblib'))
print(f"Preprocessors saved to '{output_dir}' directory.")

Shape of processed conditions: (94579, 7)
Shape of processed design data: (94579, 3)

Preprocessing complete.
Preprocessors saved to 'models' directory.


In [16]:
# --- Model Hyperparameters ---
original_dim = X_design.shape[1]
condition_dim = X_conditions.shape[1]
latent_dim = 2
intermediate_dim = 64

# --- 1. Define Encoder and Decoder as separate models ---

# ENCODER
design_input = layers.Input(shape=(original_dim,), name='design_input')
condition_input_enc = layers.Input(shape=(condition_dim,), name='condition_input_enc')
encoder_concat = layers.concatenate([design_input, condition_input_enc])
x = layers.Dense(intermediate_dim, activation='relu')(encoder_concat)
x = layers.Dense(intermediate_dim // 2, activation='relu')(x)
z_mean = layers.Dense(latent_dim, name='z_mean')(x)
z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
encoder = Model([design_input, condition_input_enc], [z_mean, z_log_var], name='encoder')
encoder.summary()

# SAMPLING LAYER
class Sampling(layers.Layer):
    """Uses (z_mean, z_log_var) to sample z, the vector encoding a digit."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# DECODER
latent_input = layers.Input(shape=(latent_dim,), name='latent_input')
condition_input_dec = layers.Input(shape=(condition_dim,), name='condition_input_dec')
decoder_concat = layers.concatenate([latent_input, condition_input_dec])
x = layers.Dense(intermediate_dim // 2, activation='relu')(decoder_concat)
x = layers.Dense(intermediate_dim, activation='relu')(x)
decoder_output = layers.Dense(original_dim, activation='sigmoid')(x)
decoder = Model([latent_input, condition_input_dec], decoder_output, name='decoder')
decoder.summary()


# --- 2. Define the VAE as a custom Model class ---
class CVAE(Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(CVAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.sampler = Sampling()
        self.total_loss_tracker = tf.keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def train_step(self, data):
        # The input data is a tuple of (design_data, condition_data)
        design_data, condition_data = data[0]

        with tf.GradientTape() as tape:
            # Encode the data to get latent space parameters
            z_mean, z_log_var = self.encoder([design_data, condition_data])
            # Sample from the latent space
            z = self.sampler([z_mean, z_log_var])
            # Decode the latent vector to reconstruct the design
            reconstruction = self.decoder([z, condition_data])

            # Calculate the losses
            reconstruction_loss = tf.reduce_mean(
                tf.square(design_data - reconstruction)
            ) * original_dim
            
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            
            total_loss = reconstruction_loss + kl_loss

        # Compute and apply gradients
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        
        # Update and return metrics
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ design_input        │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ condition_input_enc │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 10)        │          0 │ design_input[0][… │
│ (Concatenate)       │                   │            │ condition_input_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 64)        │        704 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 32)        │      2,080 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_mean (Dense)      │ (None, 2)         │         66 │ dense_11[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_log_var (Dense)   │ (None, 2)         │         66 │ dense_11[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,916 (11.39 KB)

 Trainable params: 2,916 (11.39 KB)

 Non-trainable params: 0 (0.00 B)

Model: "decoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ latent_input        │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ condition_input_dec │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 9)         │          0 │ latent_input[0][… │
│ (Concatenate)       │                   │            │ condition_input_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 32)        │        320 │ concatenate_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 64)        │      2,112 │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 3)         │        195 │ dense_13[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,627 (10.26 KB)

 Trainable params: 2,627 (10.26 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# --- Instantiate and Compile ---
# Instantiate the VAE model
cvae = CVAE(encoder, decoder)

# Compile the model with an optimizer. The loss is handled internally.
cvae.compile(optimizer=tf.keras.optimizers.Adam())

print("\nStarting model training...")

# The model now expects the input data in a slightly different format for the train_step
# We pass X as a tuple (design, conditions) and y is not needed.
history = cvae.fit(
    (X_design, X_conditions),
    epochs=200,
    batch_size=128
)

print("Training complete.")


Starting model training...
Epoch 1/200
739/739 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - kl_loss: 7.7153e-04 - loss: 0.2339 - reconstruction_loss: 0.2331
Epoch 2/200
739/739 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - kl_loss: 3.7375e-05 - loss: 0.2290 - reconstruction_loss: 0.2289
Epoch 3/200
739/739 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - kl_loss: 2.3267e-05 - loss: 0.2288 - reconstruction_loss: 0.2287
Epoch 4/200
739/739 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - kl_loss: 1.5562e-05 - loss: 0.2287 - reconstruction_loss: 0.2286
Epoch 5/200
739/739 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - kl_loss: 1.7069e-05 - loss: 0.2286 - reconstruction_loss: 0.2285
Epoch 6/200
739/739 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - kl_loss: 1.3174e-05 - loss: 0.2285 - reconstruction_loss: 0.2285
Epoch 7/200
739/739 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - kl_loss: 8.9795e-06 - loss: 0.2284 - reconstruction_loss: 0.2284
Epoch 8/200
739/739 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - kl_loss: 1.1786e-05 - loss: 0.2284 - reconstruction_loss: 0.2284
Epoch 9/200


In [19]:
# Save the entire decoder model in the recommended .keras format
decoder.save(os.path.join(output_dir, 'decoder_model.keras'))

print(f"Decoder model saved successfully to '{output_dir}/decoder_model.keras'")
print("\nPhase 2 is complete! We are now ready to build the generator application.")

Decoder model saved successfully to 'models/decoder_model.keras'

Phase 2 is complete! We are now ready to build the generator application.
